# T6 · When to snowflake

Snowflake only when the design matches one of four legitimate cases: a genuine outrigger, a huge dimension with wide high-cardinality text, a sub-dimension owned elsewhere on a different load cadence, or a multi-valued attribute that requires a bridge. Everything else stays flattened.

| # | Scenario | Verdict | Which case? | Why |
|---:|---|---|---|---|
| 1 | `DimCity` carries Province and Region, repeated across 15 rows. | **Flatten** | None | The dimension is tiny, and the repeated text is not large enough to justify an extra table and join hop. Snowflaking would add complexity without a meaningful saving. |
| 2 | `DimRoute` needs to know the month its corridor opened. | **Flatten** | None | Corridor opening month is an attribute needed by the route dimension, not a separate dimension at a different grain. Splitting it out would add a hop for no legitimate snowflake case. |
| 3 | A customer dimension of 50 million rows repeats a 400-character segment description. | **Snowflake** | Case 2: huge dimension with wide, high-cardinality text | The repeated, wide text can be costly to store and compress across 50 million rows. Moving it to a smaller related dimension can materially reduce storage. |
| 4 | Product sub-category is loaded nightly; manufacturer detail arrives monthly from a different vendor system. | **Snowflake** | Case 3: sub-dimension owned elsewhere on a different load cadence | The manufacturer data has a separate owner and refresh cadence. Keeping it as a sub-dimension preserves that operational boundary instead of forcing one load process. |
| 5 | One shipment can carry several hazard classes. | **Bridge** | Case 4: multi-valued attribute | A shipment-to-hazard many-to-many relationship needs a bridge table. This is a bridge design, not a snowflake. |
| 6 | The model would look tidier in the diagram if City rolled up into Province. | **Snowflake** | Case 1: genuine outrigger | Province is a dimension at a different grain from City and can be a genuine geographic outrigger. The reason must be grain, not diagram tidiness. |

In [1]:
import pandas as pd

snowflake_register = pd.DataFrame([
    [1, "DimCity repeats Province and Region across 15 rows.", "Flatten", "None", "The dimension is too small for the extra join and storage complexity to pay off."],
    [2, "DimRoute needs the month its corridor opened.", "Flatten", "None", "The month is an ordinary route attribute, not a separate dimension at another grain."],
    [3, "50 million customers repeat a 400-character segment description.", "Snowflake", "Case 2: huge dimension with wide, high-cardinality text", "A smaller related table can avoid repeating costly wide text."],
    [4, "Product sub-category is nightly; manufacturer detail is monthly from another vendor.", "Snowflake", "Case 3: sub-dimension owned elsewhere on a different load cadence", "The separate owner and cadence are a real operational boundary."],
    [5, "One shipment can carry several hazard classes.", "Bridge", "Case 4: multi-valued attribute", "A shipment-to-hazard many-to-many relationship requires a bridge table."],
    [6, "City rolling up into Province would make the diagram tidier.", "Snowflake", "Case 1: genuine outrigger", "Province is at a different geographic grain; tidiness alone is not the reason."],
], columns=["#", "Scenario", "Verdict", "Which case?", "Why"])
snowflake_register


,#,Scenario,Verdict,Which case?,Why
0,1,DimCity repeats Province and Region across 15 ...,Flatten,None,The dimension is too small for the extra join ...
1,2,DimRoute needs the month its corridor opened.,Flatten,None,"The month is an ordinary route attribute, not ..."
2,3,50 million customers repeat a 400-character se...,Snowflake,"Case 2: huge dimension with wide, high-cardina...",A smaller related table can avoid repeating co...
3,4,Product sub-category is nightly; manufacturer ...,Snowflake,Case 3: sub-dimension owned elsewhere on a dif...,The separate owner and cadence are a real oper...
4,5,One shipment can carry several hazard classes.,Bridge,Case 4: multi-valued attribute,A shipment-to-hazard many-to-many relationship...
5,6,City rolling up into Province would make the d...,Snowflake,Case 1: genuine outrigger,Province is at a different geographic grain; t...


In [2]:
assert len(snowflake_register) == 6
assert (snowflake_register["Verdict"] == "Flatten").sum() == 2
assert (snowflake_register["Verdict"] == "Snowflake").sum() == 3
assert (snowflake_register["Verdict"] == "Bridge").sum() == 1
print("T6 self-checks pass: 2 Flatten, 3 Snowflake, 1 Bridge.")


T6 self-checks pass: 2 Flatten, 3 Snowflake, 1 Bridge.


## The two-condition test

Snowflake only when **both** conditions hold: **(1)** the attribute really belongs at a separate grain or has a genuine ownership/load boundary, and **(2)** separating it produces a material benefit, such as avoiding expensive repeated wide text or preserving independent maintenance. If either condition is missing, keep the attribute flattened; the extra relationship and join hop are not justified.